# Notebook 2 — Preprocesamiento y features con Spark

Objetivo: transformar el dataset *Low Carbon London* en un conjunto de datos listo para entrenamiento de un modelo de forecasting de consumo eléctrico a 30 minutos.

## 1. Librerías y configuración

In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


## 2. Rutas del proyecto

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

ZIP_PATH = DATA_DIR / "LCL_Data.zip"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ZIP_PATH:", ZIP_PATH)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


## 3. Iniciar Spark en modo local

In [ ]:
spark = (
    SparkSession.builder
    .appName("LowCarbonLondon_Preprocessing")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## 4. Explorar archivos disponibles en el ZIP

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    files = z.namelist()

csv_files = [f for f in files if f.lower().endswith(".csv")]
print("Número total de archivos:", len(files))
print("Número de CSV:", len(csv_files))
print("Primeros archivos:")
for f in csv_files[:10]:
    print(" -", f)


## 5. Extraer una muestra de archivos con barra de progreso

Primero se procesa una muestra pequeña para validar el pipeline.  
Después podrás cambiar `N_SAMPLE_FILES` por un número mayor o por todos los archivos.

In [ ]:
N_SAMPLE_FILES = 10  # Cambia esto a 168 si quieres procesar todo el dataset

sample_files = csv_files[:N_SAMPLE_FILES]

# Limpiar extracción previa
for csv_path in RAW_DIR.glob("*.csv"):
    csv_path.unlink()

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(sample_files, desc="Extrayendo CSV de muestra"):
        target_path = RAW_DIR / Path(member).name
        with z.open(member) as src, open(target_path, "wb") as dst:
            dst.write(src.read())

print("Archivos extraídos:", len(sample_files))
print("Directorio:", RAW_DIR)


## 6. Leer los CSV extraídos con Spark

In [ ]:
csv_paths = sorted(str(p) for p in RAW_DIR.glob("*.csv"))

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(csv_paths)
)

print("Filas cargadas:", df.count())
df.printSchema()


## 7. Limpieza básica y normalización

In [ ]:
# Limpiar nombres de columnas
for col_name in df.columns:
    clean_name = col_name.strip()
    if clean_name != col_name:
        df = df.withColumnRenamed(col_name, clean_name)

# Convertir tipos
df = (
    df.withColumn("DateTime", F.to_timestamp(F.col("DateTime")))
      .withColumn("KWH_hh", F.regexp_replace(F.col("KWH/hh (per half hour)"), " ", ""))
      .withColumn("KWH_hh", F.col("KWH_hh").cast(DoubleType()))
      .drop("KWH/hh (per half hour)")
)

# Eliminar filas con valores faltantes en campos clave
df = df.dropna(subset=["LCLid", "stdorToU", "DateTime", "KWH_hh"])

# Orden temporal
df = df.orderBy("LCLid", "DateTime")

print("Filas después de limpieza:", df.count())
df.printSchema()


## 8. Ingeniería de características temporales

In [ ]:
df = (
    df.withColumn("hour", F.hour("DateTime"))
      .withColumn("weekday", F.date_format("DateTime", "u").cast("int") - 1)  # Monday=0
      .withColumn("month", F.month("DateTime"))
      .withColumn("is_weekend", F.when(F.col("weekday").isin([5, 6]), 1).otherwise(0))
)

df.select("LCLid", "DateTime", "hour", "weekday", "month", "is_weekend", "KWH_hh").show(10, truncate=False)


## 9. Crear variables rezagadas (lags)

In [ ]:
window_by_house = Window.partitionBy("LCLid").orderBy("DateTime")

df = (
    df.withColumn("lag_1", F.lag("KWH_hh", 1).over(window_by_house))
      .withColumn("lag_2", F.lag("KWH_hh", 2).over(window_by_house))
      .withColumn("lag_48", F.lag("KWH_hh", 48).over(window_by_house))     # 24 horas atrás
      .withColumn("lag_336", F.lag("KWH_hh", 336).over(window_by_house))   # 7 días atrás
)

df.select(
    "LCLid", "DateTime", "KWH_hh", "lag_1", "lag_2", "lag_48", "lag_336"
).show(10, truncate=False)


## 10. Construir la variable objetivo

In [ ]:
df = df.withColumn("target", F.lead("KWH_hh", 1).over(window_by_house))

df.select("LCLid", "DateTime", "KWH_hh", "target").show(10, truncate=False)


## 11. Dataset final para modelado

In [ ]:
model_df = df.dropna(subset=["target", "lag_1", "lag_2", "lag_48", "lag_336"])

print("Filas listas para modelado:", model_df.count())
model_df.printSchema()


## 12. Visualización relevante para el reporte

In [ ]:
# Perfil promedio de consumo por hora (sirve para justificar la feature hour)
hourly_profile = (
    model_df.groupBy("hour")
    .agg(F.avg("KWH_hh").alias("avg_consumption"))
    .orderBy("hour")
    .toPandas()
)

plt.figure(figsize=(10, 5))
plt.plot(hourly_profile["hour"], hourly_profile["avg_consumption"], marker="o")
plt.title("Perfil promedio de consumo por hora")
plt.xlabel("Hora del día")
plt.ylabel("Consumo promedio (kWh)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Interpretación:** esta gráfica muestra que la hora del día sí está relacionada con el consumo promedio, por lo que `hour` se incorpora como variable explicativa en el modelo de forecasting.

## 13. Guardar el dataset procesado en Parquet

In [ ]:
output_path = str(PROCESSED_DIR / "lcl_preprocessed_parquet")

(
    model_df
    .drop("stdorToU")  # En la muestra explorada solo aparece "Std"
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print("Dataset procesado guardado en:")
print(output_path)
print("=== PREPROCESAMIENTO COMPLETADO ===")


## 14. Cerrar Spark

In [ ]:
spark.stop()
print("Spark detenido correctamente.")
